In [ ]:
# ==============================
# PHASE 03: SEQUENCE BUILDING
# (Strict Roadmap Compliance: Sliding Window + Normalization)
# ==============================

import os
import numpy as np
from tqdm.notebook import tqdm
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. PATHS
PROJECT_ROOT = "/content/drive/MyDrive/Human-Fall-Recognition"
SKELETON_ROOT = os.path.join(PROJECT_ROOT, "Skeleton_Raw")
SEQ_OUT = os.path.join(PROJECT_ROOT, "Sequences")
os.makedirs(SEQ_OUT, exist_ok=True)

CLASSES = ["NoFall", "Faint", "Slip", "Trip"]
LABEL_MAP = {c:i for i,c in enumerate(CLASSES)}

# --- CRITICAL CONFIGURATION ---
SEQ_LEN = 50     # 50 frames (approx 1.5 - 2.0 seconds of real time)
STRIDE = 10      # Create a new sequence every 10 frames (Overlapping windows)

X, y, video_ids = [], [], []

# 3. NORMALIZATION FUNCTION (Hip-Center)
def normalize(seq):
    # seq shape: (Frames, 17, 3) -> (x, y, conf)
    xy = seq[..., :2]
    conf = seq[..., 2:]

    # keypoint 11=Left Hip, 12=Right Hip. We take the average as "Center"
    # Handle cases where keypoints might be missing (use average of all points as fallback)
    if xy.shape[1] > 11:
        hip = (xy[:, 11:12, :] + xy[:, 12:13, :]) / 2
    else:
        hip = xy.mean(axis=1, keepdims=True)

    # Subtract hip center to make movement "Camera Independent"
    xy = xy - hip
    return np.concatenate([xy, conf], axis=2)

# 4. BUILD SEQUENCES
print("--------------------------------------------------")
print("PHASE 3: BUILDING SEQUENCES (SLIDING WINDOW)")
print("--------------------------------------------------")

for cls in CLASSES:
    cls_dir = os.path.join(SKELETON_ROOT, cls)
    if not os.path.exists(cls_dir):
        print(f"⚠️ Warning: Folder {cls} not found")
        continue

    files = [f for f in os.listdir(cls_dir) if f.endswith(".npy")]
    print(f"Processing {cls}: {len(files)} videos...")

    for f in tqdm(files, desc=cls):
        try:
            # Load Raw Skeleton: (Total_Frames, 17, 3)
            file_path = os.path.join(cls_dir, f)
            data = np.load(file_path)

            if len(data) < SEQ_LEN:
                # OPTIONAL: Pad short videos (or skip them).
                # For high quality, we usually skip videos shorter than 1 window (50 frames)
                # But to follow "All Frames Used" vaguely, let's pad.
                pad = np.zeros((SEQ_LEN - len(data), 17, 3))
                data = np.concatenate([data, pad], axis=0)

            # Normalize entire video first
            data = normalize(data)

            # SLIDING WINDOW (Stride 10)
            # This creates MULTIPLE sequences from ONE video (Data Augmentation)
            num_frames = data.shape[0]
            for start in range(0, num_frames - SEQ_LEN + 1, STRIDE):
                window = data[start : start + SEQ_LEN] # Shape: (50, 17, 3)

                # Flatten for LSTM: (50, 51)
                window = window.reshape(SEQ_LEN, -1)

                X.append(window)
                y.append(LABEL_MAP[cls])
                video_ids.append(f.replace(".npy", "")) # Keep ID for voting later

        except Exception as e:
            print(f"Error processing {f}: {e}")

# 5. SAVE FINAL ARRAYS
X = np.array(X)
y = np.array(y)
video_ids = np.array(video_ids)

print("\n--------------------------------------------------")
print(f"✅ PHASE 3 COMPLETE")
print(f"Total Sequences Created: {X.shape[0]}")
print(f"Data Shape: {X.shape}")
print(f"Saved to: {SEQ_OUT}")
print("--------------------------------------------------")

np.save(os.path.join(SEQ_OUT, "X.npy"), X)
np.save(os.path.join(SEQ_OUT, "y.npy"), y)
np.save(os.path.join(SEQ_OUT, "video_ids.npy"), video_ids)